## Step 1: Downloading the CIFAR-10 dataset and splitting it into train/test sets

---

In [15]:
# Importing the necessary library
from keras.datasets import cifar10

# Downloading the CIFAR-10 dataset and splitting it into train/test sets
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Printing the dimensions to confirm the dataset loaded successfully
print("Training data (x_train) shape:", x_train.shape)
print("Training labels (y_train) shape:", y_train.shape)
print("Test data (x_test) shape:", x_test.shape)
print("Test labels (y_test) shape:", y_test.shape)

Training data (x_train) shape: (50000, 32, 32, 3)
Training labels (y_train) shape: (50000, 1)
Test data (x_test) shape: (10000, 32, 32, 3)
Test labels (y_test) shape: (10000, 1)


## Step 2: Converting Images to 1D Vector Format

---



In [16]:
# Step 2: Converting Images to 1D Vector Format
# The '-1' parameter in reshape ensures that the number of samples (50000 for train, 10000 for test)
# is automatically calculated and preserved without being distorted.

x_train = x_train.reshape(-1, 3072)
x_test = x_test.reshape(-1, 3072)

# Printing the new dimensions to verify that the operation was successful
print("Vectorized training data (x_train) shape:", x_train.shape)
print("Vectorized test data (x_test) shape:", x_test.shape)

Vectorized training data (x_train) shape: (50000, 3072)
Vectorized test data (x_test) shape: (10000, 3072)


## Step 3: Implement KNN Classification

In [17]:
import numpy as np
from collections import Counter

# CIFAR-10 Class Names
cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# Step 1: Custom Similarity Function
def custom_similarity(x, y):
    # Flattening the data and preventing data type overflow (Underflow)
    x_vec = x.flatten().astype(np.float32)
    y_vec = y.flatten().astype(np.float32)

    # Calculating the L2 (Euclidean) Distance
    l2_distance = np.linalg.norm(x_vec - y_vec)

    # Converting the distance into a "Similarity" score
    similarity = 1 / (1 + l2_distance)

    return similarity

# Step 2: KNN Classification Function
def knnCustomSimilarity(x_train, y_train, sample_test, k):
    similarities = []

    # Comparing the test sample with the training set
    for i in range(len(x_train)):
        sim = custom_similarity(sample_test, x_train[i])
        similarities.append((sim, y_train[i][0]))

    # Sorting the scores from LARGEST to SMALLEST (descending)
    similarities.sort(key=lambda item: item[0], reverse=True)

    # Selecting the first k neighbors with the highest similarity score
    top_k_neighbors = similarities[:k]

    # --- DETAILED LOG OUTPUTS ---
    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(" CLASSIFICATION STATISTICS")
    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"[INPUT DETAIL] K Parameter: {k}")
    print("\n[SORTED NEAREST NEIGHBORS]")

    # Printing each neighbor line by line in detail (with 8 decimal precision)
    for rank, (sim, label_idx) in enumerate(top_k_neighbors, 1):
        class_name = cifar10_classes[label_idx]
        print(f"Neighbor {rank} -> Similarity Score: {sim:.8f} | Class: {class_name}")

    print("\n[VOTE DISTRIBUTION (MAJORITY VOTE)]")
    top_k_labels = [neighbor[1] for neighbor in top_k_neighbors]
    vote_counts = Counter(top_k_labels)

    # Listing how many votes each class received
    for label_idx, count in vote_counts.most_common():
        print(f"- {cifar10_classes[label_idx]}: {count} votes")

    # Finding the most frequently occurring class
    most_common_label = vote_counts.most_common(1)[0][0]
    final_class = cifar10_classes[most_common_label]

    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"[OUTPUT RESULT] Predicted Class: >> {final_class.upper()} <<\n")
    # ------------------------------------

    return final_class

# Step 3: Test Phase
sample_test = x_test[0, :]
k = 5
similar_class_name = knnCustomSimilarity(x_train, y_train, sample_test, k)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 CLASSIFICATION STATISTICS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[INPUT DETAIL] K Parameter: 5

[SORTED NEAREST NEIGHBORS]
Neighbor 1 -> Similarity Score: 0.00040521 | Class: deer
Neighbor 2 -> Similarity Score: 0.00038789 | Class: bird
Neighbor 3 -> Similarity Score: 0.00038733 | Class: bird
Neighbor 4 -> Similarity Score: 0.00038583 | Class: deer
Neighbor 5 -> Similarity Score: 0.00038559 | Class: bird

[VOTE DISTRIBUTION (MAJORITY VOTE)]
- bird: 3 votes
- deer: 2 votes
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[OUTPUT RESULT] Predicted Class: >> BIRD <<



## Step 4: Test

---

In [18]:
import numpy as np
import pandas as pd
from collections import Counter
from IPython.display import display

# CIFAR-10 Class Names
cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

def test_and_visualize_knn(test_index, k=5):
    # 1. Getting the test sample and its corresponding true label
    sample_test = x_test[test_index]
    actual_class = cifar10_classes[y_test[test_index][0]]

    # 2. Vectorized calculation of the L2 (Euclidean) distance between the test sample and the entire training set
    # Note: astype(np.float32) is used to prevent data type overflow (Integer Underflow).
    distances = np.linalg.norm(x_train.astype(np.float32) - sample_test.astype(np.float32), axis=1)

    # 3. Converting the distance value to a "Custom Similarity" score in accordance with the HW2 instructions
    similarities = 1 / (1 + distances)

    # 4. Creating a Pandas DataFrame to display the results in a tabular format
    df = pd.DataFrame({
        'Training Sample Index': range(len(x_train)),
        'Distance (L2)': distances,
        'Similarity Score': similarities,
        'Class Name': [cifar10_classes[label[0]] for label in y_train]
    })

    # 5. Sorting the table in descending order (largest to smallest) based on the "Similarity Score" according to the HW2 instructions
    df_sorted = df.sort_values(by='Similarity Score', ascending=False).reset_index(drop=True)

    # 6. Selecting the first K neighbors with the highest similarity (Top K)
    top_k_df = df_sorted.head(k)

    # 7. Performing the majority vote process for classification
    predicted_class = Counter(top_k_df['Class Name']).most_common(1)[0][0]

    # Printing the results to the screen in a formatted way
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f" TEST SAMPLE INFO (Index: {test_index})")
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"True Class     : {actual_class}")
    print(f"Selected K     : {k}")
    print(f"\n[Sorted Top {k} Nearest Neighbors Table]")

    # Visualizing the table in the Colab environment via the display() function
    display(top_k_df)

    print(f"\n=> MAJORITY VOTE RESULT: {predicted_class}")

    # Providing a status notification based on the accuracy of the prediction
    if predicted_class == actual_class:
        print("=> STATUS: ✅ SUCCESSFUL PREDICTION")
    else:
        print("=> STATUS: ❌ INCORRECT PREDICTION")
    print("\n\n")


# ---------------------------------------------------------
# TESTING PHASE WITH DIFFERENT DATA
# ---------------------------------------------------------

# Determining the indices to be tested from the x_test dataset
# Sample test indices: 0, 12, 45, and 100
test_indexes = [0, 12, 45, 100]

# Defining the number of K neighbors parametrically
choosed_k_value = 10

for idx in test_indexes:
    test_and_visualize_knn(test_index=idx, k=choosed_k_value)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 TEST SAMPLE INFO (Index: 0)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
True Class     : cat
Selected K     : 10

[Sorted Top 10 Nearest Neighbors Table]


,Training Sample Index,Distance (L2),Similarity Score,Class Name
0,47188,2466.830322,0.000405,deer
1,20051,2577.049316,0.000388,bird
2,17972,2580.759277,0.000387,bird
3,43234,2590.820068,0.000386,deer
4,31377,2592.415039,0.000386,bird
5,20447,2606.589111,0.000383,cat
6,8309,2608.596436,0.000383,frog
7,9389,2624.265625,0.000381,deer
8,420,2641.434082,0.000378,deer
9,35193,2642.295898,0.000378,dog



=> MAJORITY VOTE RESULT: deer
=> STATUS: ❌ INCORRECT PREDICTION



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 TEST SAMPLE INFO (Index: 12)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
True Class     : dog
Selected K     : 10

[Sorted Top 10 Nearest Neighbors Table]


,Training Sample Index,Distance (L2),Similarity Score,Class Name
0,8398,2417.640869,0.000413,dog
1,2745,2492.876953,0.000401,dog
2,3540,2498.911621,0.000400,deer
3,40169,2499.500732,0.000400,ship
4,17939,2502.595947,0.000399,cat
5,9762,2503.009521,0.000399,frog
6,4124,2519.588379,0.000397,cat
7,48889,2521.620117,0.000396,deer
8,528,2535.627930,0.000394,deer
9,46423,2539.389160,0.000394,cat



=> MAJORITY VOTE RESULT: deer
=> STATUS: ❌ INCORRECT PREDICTION



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 TEST SAMPLE INFO (Index: 45)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
True Class     : truck
Selected K     : 10

[Sorted Top 10 Nearest Neighbors Table]


,Training Sample Index,Distance (L2),Similarity Score,Class Name
0,25002,2850.777832,0.000351,ship
1,18379,2936.496826,0.000340,ship
2,26852,2970.052490,0.000337,truck
3,46769,2971.059082,0.000336,deer
4,1759,3005.631104,0.000333,airplane
5,22340,3009.014404,0.000332,ship
6,34948,3017.279297,0.000331,airplane
7,9648,3018.569336,0.000331,ship
8,49528,3021.975586,0.000331,airplane
9,8437,3022.429443,0.000331,airplane



=> MAJORITY VOTE RESULT: ship
=> STATUS: ❌ INCORRECT PREDICTION



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 TEST SAMPLE INFO (Index: 100)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
True Class     : deer
Selected K     : 10

[Sorted Top 10 Nearest Neighbors Table]


,Training Sample Index,Distance (L2),Similarity Score,Class Name
0,16640,1994.939575,0.000501,deer
1,14313,2002.915894,0.000499,cat
2,40085,2048.096191,0.000488,deer
3,48458,2118.445312,0.000472,airplane
4,29415,2127.122314,0.000470,bird
5,25254,2129.477051,0.000469,cat
6,46896,2154.707275,0.000464,horse
7,38773,2157.641602,0.000463,frog
8,21171,2159.271240,0.000463,deer
9,40860,2169.499023,0.000461,bird



=> MAJORITY VOTE RESULT: deer
=> STATUS: ✅ SUCCESSFUL PREDICTION



